# The 14th Homework

## Импорты, seed и среда

In [1]:
import os
import random
from typing import List, Dict, Tuple, Optional
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_gigachat import GigaChat
import json
import matplotlib
import matplotlib.pyplot as plt
import plotly.graph_objects as go

ARTIFACTS_DIR = "artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

K = 5

import numpy as np
import pandas as pd
import torch

import re
from pathlib import Path
from collections import Counter

import frontmatter

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

os.environ["TOKENIZERS_PARALLELISM"] = "false"

FAISS_AVAILABLE = True


print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS available:", FAISS_AVAILABLE)
KB_DIR = Path("./knowledge_base/skincare_kb")

/Users/polina/vscode/art_ing_course/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NumPy: 2.4.4
Pandas: 3.0.2
FAISS available: True


In [2]:
SEED = 42
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps"  if torch.backends.mps.is_available() else
    "cpu"
)

print("Устройство для работы:", DEVICE)

Устройство для работы: mps


## База знаний и первичный анализ

In [3]:
def load_kb(kb_dir: str) -> pd.DataFrame:
    records = []
    path = Path(kb_dir)

    for md_file in sorted(path.rglob("*.md")):
        post = frontmatter.load(md_file)       # парсим YAML + тело
        meta = post.metadata
        body = post.content.strip()

        records.append({
            # путь
            "filename":    md_file.name,
            "rel_path":    str(md_file.relative_to(path)),
            # метаданные из YAML
            "title":       meta.get("title", ""),
            "category":    meta.get("category", ""),
            "subcategory": meta.get("subcategory", ""),
            "tags":        meta.get("tags", []),
            "skin_type":   meta.get("skin_type", []),
            "skin_concern":meta.get("skin_concern", []),
            # статистика
            "body":        body,
            "n_words":     len(body.split()),
            "n_chars":     len(body),
            "n_h2":        len(re.findall(r"^##\s+", body, re.M)),
        })

    return pd.DataFrame(records)

In [4]:
df = load_kb(KB_DIR)

print(f"Загружено документов: {len(df)}")
df.sample(10)

Загружено документов: 33


,filename,rel_path,title,category,subcategory,tags,skin_type,skin_concern,body,n_words,n_chars,n_h2
31,01_sos_skin_detox.md,06_procedures_and_techniques/01_sos_skin_detox.md,SOS-детокс кожи после праздников,процедуры,ситуативный_уход,"[детокс, восстановление, отеки, воспаления, об...",[все типы],"[обезвоживание, тусклость, отеки, воспаления]",# SOS-Детокс для кожи (после вечеринок и празд...,151,1062,3
15,03_reading_inci_labels.md,02_cosmetics_and_ingredients/03_reading_inci_l...,Чтение этикеток и составов (INCI),косметика,состав_и_этикетки,"[inci, этикетки, состав, ингредиенты, безопасн...",[все типы],[],# Чтение этикеток и составов (INCI)\n\n## Прав...,289,2161,1
26,02_curly_girl_method.md,04_hair_care/02_curly_girl_method.md,Метод Curly Girl: уход за кудрявыми волосами,уход_за_волосами,методы_ухода,"[curly_girl, кудрявые_волосы, метод_кудряшки, ...",[],[],# Curly Girl Method\n\n## Что такое метод Curl...,239,1594,5
17,02_oily_skin.md,03_skincare_by_type_and_concern/01_skin_types/...,Жирная кожа,уход_за_кожей,типы_кожи,"[жирная_кожа, себум, поры, матирование, акне]",[жирная],"[акне, поры, тусклость]",# Oily Skin\n\n## Что такое жирная кожа\n\nЖир...,366,2463,5
8,04_spf_protection.md,02_cosmetics_and_ingredients/01_cosmetic_produ...,Защита от солнца (SPF),косметика,типы_средств,"[spf, солнцезащита, uva, uvb, фотостарение, пи...",[все типы],"[пигментация, морщины, фотостарение]",# Защита от солнца (SPF)\n\n## Правила использ...,264,1947,1
9,01_retinol_and_derivatives.md,02_cosmetics_and_ingredients/02_active_ingredi...,Ретинол и его производные,косметика,активные_ингредиенты,"[ретинол, ретиноиды, антиэйдж, акне, витамин_А]","[жирная, комбинированная, зрелая]","[акне, морщины, пигментация, поры]",# Ретинол и его производные (Ретиноиды)\n\n## ...,256,2017,4
19,04_sensitive_skin.md,03_skincare_by_type_and_concern/01_skin_types/...,Чувствительная кожа: реакции и уход,уход_за_кожей,типы_кожи,"[чувствительная_кожа, раздражение, барьер, куп...",[чувствительная],"[купероз, розацеа, раздражение, сухость]",# Реакция кожи на холод: Светофор состояний и ...,270,2209,3
21,02_pigmentation.md,03_skincare_by_type_and_concern/02_skin_concer...,Пигментация,уход_за_кожей,проблемы_кожи,"[пигментация, мелазма, осветление, витамин_с, ...",[все типы],"[пигментация, тусклость, постакне]",# Pigmentation\n\n## Что такое пигментация\n\n...,411,2941,7
12,04_copper_peptides.md,02_cosmetics_and_ingredients/02_active_ingredi...,Пептиды меди в косметике,косметика,активные_ингредиенты,"[пептиды, пептиды_меди, антиэйдж, коллаген, ан...","[зрелая, нормальная, комбинированная]","[морщины, потеря_упругости, воспаления]",# Пептиды меди в косметике (Copper Peptides)\n...,172,1251,4
0,01_healthy_eating_basics.md,01_nutrition_and_diets/01_healthy_eating_basic...,Основы здорового питания для красоты и здоровья,питание,основы,"[питание, рацион, здоровье, антиоксиданты, вит...",[все типы],"[тусклость, сухость, акне]",# Основы здорового питания для красоты и здоро...,196,1486,1


In [5]:
df[df["n_chars"] < 500][["filename", "n_chars"]]

,filename,n_chars


In [6]:
df[df["title"] == ""][["filename", "rel_path"]]

,filename,rel_path


In [7]:
df[df["title"].duplicated(keep=False)][["filename", "title"]]

,filename,title


In [8]:
df[df["n_h2"] == 0][["filename", "category"]]

,filename,category


In [9]:
for i in range (5):
    print(f"title: {df['title'].iloc[i]} \ntext - {df['body'][:10]}")

title: Основы здорового питания для красоты и здоровья 
text - 0    # Основы здорового питания для красоты и здоро...
1    # Продукты, полезные для кожи\n#полезные_проду...
2    # Продукты, вредные для кожи\n#вредные_продукт...
3    # Питьевой режим и гидратация\n#питьевой_режим...
4    # Диета при акне (Anti-acne протокол)\n#диета_...
5    # Очищение кожи: демакияж и умывание\n#очищени...
6    # Тонизирование: Тоник, Тонер и Эссенция\n\n##...
7    # Увлажнение кожи: кремы и сыворотки\n#увлажня...
8    # Защита от солнца (SPF)\n\n## Правила использ...
9    # Ретинол и его производные (Ретиноиды)\n\n## ...
Name: body, dtype: str
title: Продукты, полезные для кожи 
text - 0    # Основы здорового питания для красоты и здоро...
1    # Продукты, полезные для кожи\n#полезные_проду...
2    # Продукты, вредные для кожи\n#вредные_продукт...
3    # Питьевой режим и гидратация\n#питьевой_режим...
4    # Диета при акне (Anti-acne протокол)\n#диета_...
5    # Очищение кожи: демакияж и умывание\n#оч

База знаний корректна. Выбрана предметная область - уход за внешностью. База знаний полностью покрывает возможные вопросы по этой теме, при этом вписывается в размер. Также в базе знаний настроен фильтр по категориям, заголовки и тп

## Чанкинг

In [10]:
HEADERS_TO_SPLIT = [
    ("#",   "h1"),   # заголовок документа
    ("##",  "h2"),   # основные секции → главные чанки
    ("###", "h3"),   # подсекции → дочерние чанки
]
MAX_CHUNK_SIZE = 1000  
CHUNK_OVERLAP  = 100   

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=HEADERS_TO_SPLIT,
    strip_headers=False   # заголовок остаётся в тексте чанка
)

# Страховочный сплиттер для слишком длинных секций
char_splitter = RecursiveCharacterTextSplitter(
    chunk_size=MAX_CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

In [11]:
all_chunks = []

for _, row in df.iterrows():
    # 1. Режем по заголовкам ##
    header_chunks = md_splitter.split_text(row["body"])

    for chunk in header_chunks:
        # 2. Если секция длиннее MAX_CHUNK_SIZE — дробим дополнительно
        if len(chunk.page_content) > MAX_CHUNK_SIZE:
            sub_chunks = char_splitter.split_documents([chunk])
        else:
            sub_chunks = [chunk]

        for sc in sub_chunks:
            # 3. Добавляем метаданные из YAML-шапки
            sc.metadata.update({
                "source":      row["filename"],
                "title":       row["title"],
                "category":    row["category"],
                "subcategory": row["subcategory"],
                "skin_type":   row["skin_type"],
                "skin_concern":row["skin_concern"],
                "tags":        row["tags"],
            })
            all_chunks.append(sc)

print(f"Документов:  {len(df)}")
print(f"Чанков всего: {len(all_chunks)}")
print(f"Среднее на документ: {len(all_chunks)/len(df):.1f}")

Документов:  33
Чанков всего: 182
Среднее на документ: 5.5


In [12]:
# Выбираем один документ для наглядности
doc_name = '01_skin_beneficial_foods.md'
doc_chunks = [c for c in all_chunks if c.metadata["source"] == doc_name]

print(f"📄 {doc_name} → {len(doc_chunks)} чанков\n")
for i, c in enumerate(doc_chunks):
    section = c.metadata.get("h2", c.metadata.get("h1", "—"))
    print(f"  Чанк {i+1}: [{section}]  {len(c.page_content)} симв.")
    print(f"  {c.page_content[:120].strip()}...")
    print()

📄 01_skin_beneficial_foods.md → 3 чанков

  Чанк 1: [Влияние продуктов на состояние кожи]  324 симв.
  # Продукты, полезные для кожи
#полезные_продукты, #продукты_для_кожи, #омега3, #антиоксиданты, #коллаген_из_пищи  
## Вл...

  Чанк 2: [Влияние продуктов на состояние кожи]  898 симв.
  ### Топ-компоненты и их пищевые источники
- **Омега-3 жирные кислоты:** Снимают воспаление, укрепляют клеточные мембраны...

  Чанк 3: [Влияние продуктов на состояние кожи]  286 симв.
  ### Рекомендации для внедрения
- Добавляйте порцию жирной рыбы 2-3 раза в неделю.
- Ежедневно употребляйте минимум 400 г...



## Эмбеддинги и индекс FAISS

In [13]:
embedding_model = HuggingFaceEmbeddings(
    model_name="deepvk/USER-base",
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True},  # обязательно для cosine similarity
)

# Быстрая проверка
test_vec = embedding_model.embed_query("тест")
print(f"Размерность вектора: {len(test_vec)}")  # → 768

Loading weights: 100%|██████████| 160/160 [00:00<00:00, 7179.11it/s]
Default prompt name is set to 'query'. This prompt will be applied to all `encode()` calls, except if `encode()` is called with `prompt` or `prompt_name` parameters.


Размерность вектора: 768


In [ ]:
def normalize_metadata(meta: dict) -> dict:
    """FAISS/LangChain требует строки в metadata, не списки"""
    result = {}
    for k, v in meta.items():
        if isinstance(v, list):
            result[k] = ", ".join(str(i) for i in v)
        else:
            result[k] = str(v) if v is not None else ""
    return result

# Нормализуем metadata в каждом чанке
for chunk in all_chunks:
    chunk.metadata = normalize_metadata(chunk.metadata)

print(f"Чанков готово к индексации: {len(all_chunks)}")
print(f"\nПример метаданных чанка:")
for k, v in all_chunks[0].metadata.items():
    print(f"  {k}: {v}")

Чанков готово к индексации: 182

Пример метаданных чанка:
  h1: Основы здорового питания для красоты и здоровья
  h2: Базовые принципы нутрициологии
  source: 01_healthy_eating_basics.md
  title: Основы здорового питания для красоты и здоровья
  category: питание
  subcategory: основы
  skin_type: все типы
  skin_concern: тусклость, сухость, акне
  tags: питание, рацион, здоровье, антиоксиданты, витамины


In [15]:
from langchain_community.vectorstores import FAISS

print("Строим индекс... (займёт ~30–60 сек)")

vectorstore = FAISS.from_documents(
    documents=all_chunks,
    embedding=embedding_model,
)

# Сохраняем на диск — чтобы не пересчитывать каждый раз
vectorstore.save_local("./faiss_index")
print(f"✅ Индекс сохранён: ./faiss_index/")
print(f"   Векторов в индексе: {vectorstore.index.ntotal}")

Строим индекс... (займёт ~30–60 сек)
✅ Индекс сохранён: ./faiss_index/
   Векторов в индексе: 182


In [16]:
# Прямой поиск по FAISS-индексу
query_vec = np.array(embedding_model.embed_query("тест"), dtype="float32").reshape(1, -1)
distances, indices = vectorstore.index.search(query_vec, k=5)
print("FAISS distances:", distances)
print("FAISS indices:", indices)

FAISS distances: [[1.3358943 1.3391452 1.3477669 1.3767399 1.3801146]]
FAISS indices: [[169  84 164 139  36]]


In [17]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

queries = [
    "как ухаживать за сухой кожей зимой",
    "какие кислоты помогают от акне",
    "что такое постакне и как от него избавиться",
    "какие продукты улучшают состояние кожи",
    "как правильно наносить SPF",
]

for query in queries:
    print(f"\n{'─'*60}")
    print(f"🔍 Запрос: {query}")
    print(f"{'─'*60}")
    results = retriever.invoke(query)

    for i, doc in enumerate(results, 1):
        meta = doc.metadata
        section = meta.get("h2", meta.get("h1", "—"))
        print(f"\n  [{i}] {meta.get('title')}  →  {section}")
        print(f"      категория: {meta.get('category')} / {meta.get('subcategory')}")
        print(f"      {doc.page_content[:200].strip()}...")


────────────────────────────────────────────────────────────
🔍 Запрос: как ухаживать за сухой кожей зимой
────────────────────────────────────────────────────────────

  [1] Подготовка кожи к весне  →  Подготовка кожи к весне
      категория: процедуры / сезонный_уход
      # Подготовка кожи к весне  
Переход от плотных зимних текстур к более легким и защитным форматам из-за перепадов температур и активного солнца....

  [2] Реакция кожи на холод — зимний уход  →  Реакция кожи на холод: «Светофор» состояний и уход
      категория: уход_за_кожей / проблемы_кожи
      # Реакция кожи на холод: «Светофор» состояний и уход  
Перепады температур, мороз, ветер и сухой воздух в помещениях разрушают гидролипидную мантию кожи. Диагностика по принципу «светофора»:...

  [3] Сезонные и SOS-ритуалы ухода  →  Подготовка кожи к весне
      категория: процедуры / ритуалы
      ## Подготовка кожи к весне  
Переход от плотных зимних текстур к более легким и защитным форматам из-за перепадов температур 

In [18]:
# similarity_search_with_score возвращает L2-дистанцию
# Чем меньше — тем ближе (при normalize_embeddings=True)

query = "ретинол против морщин: с чего начать"
results_with_scores = vectorstore.similarity_search_with_score(query, k=5)

print(f"Запрос: {query}\n")
for doc, score in results_with_scores:
    print(f"  score={score:.4f}  |  {doc.metadata.get('title')}  →  {doc.metadata.get('h2','—')}")

Запрос: ретинол против морщин: с чего начать

  score=0.7316  |  Ретинол и его производные  →  Что такое ретинол и как он работает
  score=0.8479  |  Морщины и потеря упругости  →  Ошибки в антивозрастном уходе
  score=0.8946  |  Морщины и потеря упругости  →  Ключевые активы
  score=0.9026  |  Ретинол и его производные  →  Правила ввода ретинола в уход (Лестница ретинола)
  score=0.9143  |  Комбинированная кожа  →  Ингредиенты


## Контрольные запросы и оценка retrieval

In [19]:
EVAL_QUERIES = [
    {
        "id": "Q01",
        "query": "как ухаживать за сухой кожей зимой",
        "relevant_docs": ["01_dry_skin.md"],
        "relevant_keywords": ["сухая", "барьер", "церамиды", "шелушение"],
        "note": "Прямое совпадение с документом типа кожи"
    },
    {
        "id": "Q02",
        "query": "какие кислоты помогают от акне и расширенных пор",
        "relevant_docs": ["03_aha_bha_acids.md", "01_acne_and_post_acne.md"],
        "relevant_keywords": ["bha", "салициловая", "акне", "поры"],
        "note": "Два релевантных документа"
    },
    {
        "id": "Q03",
        "query": "что такое постакне и как убрать тёмные пятна",
        "relevant_docs": ["01_acne_and_post_acne.md", "02_pigmentation.md"],
        "relevant_keywords": ["постакне", "пигментация", "пятна", "PIH"],
        "note": "Два тематически близких документа"
    },
    {
        "id": "Q04",
        "query": "продукты и питание для здоровой кожи",
        "relevant_docs": ["01_skin_beneficial_foods.md", "01_healthy_eating_basics.md"],
        "relevant_keywords": ["продукты", "питание", "омега", "антиоксидант"],
        "note": "Раздел питания"
    },
    {
        "id": "Q05",
        "query": "как правильно наносить и выбирать SPF крем",
        "relevant_docs": ["04_spf_protection.md"],
        "relevant_keywords": ["spf", "солнцезащита", "uva", "uvb"],
        "note": "Узкий конкретный запрос"
    },
    {
        "id": "Q06",
        "query": "ретинол с чего начать и как избежать раздражения",
        "relevant_docs": ["01_retinol_and_derivatives.md"],
        "relevant_keywords": ["ретинол", "ретинизация", "раздражение", "концентрация"],
        "note": "Запрос про конкретный ингредиент"
    },
    {
        "id": "Q07",
        "query": "уход за жирной кожей матирование и поры",
        "relevant_docs": ["02_oily_skin.md"],
        "relevant_keywords": ["жирная", "себум", "матирование", "ниацинамид"],
        "note": "Прямое совпадение тип кожи"
    },
    {
        "id": "Q08",
        "query": "витамин С сыворотка осветление пигментных пятен",
        "relevant_docs": ["02_vitamin_c.md", "02_pigmentation.md"],
        "relevant_keywords": ["витамин с", "аскорбиновая", "осветление", "тирозиназа"],
        "note": "Смешанный: ингредиент + проблема"
    },
    {
        "id": "Q09",
        "query": "выпадение волос причины и что делать",
        "relevant_docs": ["01_hair_loss.md"],
        "relevant_keywords": ["выпадение", "алопеция", "волосы", "трихология"],
        "note": "Раздел волос — отдельная категория"
    },
    {
        "id": "Q10",
        "query": "как читать состав косметики и что такое INCI",
        "relevant_docs": ["03_reading_inci_labels.md"],
        "relevant_keywords": ["inci", "состав", "этикетка", "ингредиенты"],
        "note": "Точный узкий запрос"
    },
]

print(f"Контрольных запросов: {len(EVAL_QUERIES)}")
for q in EVAL_QUERIES:
    print(f"  {q['id']}  {q['query'][:55]:<55}  → {q['relevant_docs']}")


Контрольных запросов: 10
  Q01  как ухаживать за сухой кожей зимой                       → ['01_dry_skin.md']
  Q02  какие кислоты помогают от акне и расширенных пор         → ['03_aha_bha_acids.md', '01_acne_and_post_acne.md']
  Q03  что такое постакне и как убрать тёмные пятна             → ['01_acne_and_post_acne.md', '02_pigmentation.md']
  Q04  продукты и питание для здоровой кожи                     → ['01_skin_beneficial_foods.md', '01_healthy_eating_basics.md']
  Q05  как правильно наносить и выбирать SPF крем               → ['04_spf_protection.md']
  Q06  ретинол с чего начать и как избежать раздражения         → ['01_retinol_and_derivatives.md']
  Q07  уход за жирной кожей матирование и поры                  → ['02_oily_skin.md']
  Q08  витамин С сыворотка осветление пигментных пятен          → ['02_vitamin_c.md', '02_pigmentation.md']
  Q09  выпадение волос причины и что делать                     → ['01_hair_loss.md']
  Q10  как читать состав косметики и что такое INCI    

In [20]:
def hit_at_k(retrieved_docs, relevant_filenames, relevant_keywords, k):
    """
    Hit@k = 1 если хотя бы один из top-k чанков
    принадлежит релевантному документу ИЛИ содержит ключевые слова.
    """
    for doc in retrieved_docs[:k]:
        source = doc.metadata.get("source", "")
        content = doc.page_content.lower()
        # Проверка по имени файла
        if any(rel in source for rel in relevant_filenames):
            return 1
        # Запасная проверка по ключевым словам
        if any(kw.lower() in content for kw in relevant_keywords):
            return 1
    return 0


def recall_at_k(retrieved_docs, relevant_filenames, relevant_keywords, k):
    """
    Recall@k = доля найденных релевантных документов из всех ожидаемых.
    Считаем по именам файлов.
    """
    found = set()
    for doc in retrieved_docs[:k]:
        source = doc.metadata.get("source", "")
        content = doc.page_content.lower()
        for rel in relevant_filenames:
            if rel in source:
                found.add(rel)
        # Запасной критерий: ключевые слова
        if any(kw.lower() in content for kw in relevant_keywords):
            for rel in relevant_filenames:
                found.add(rel)  # засчитываем как покрытие
    return len(found) / len(relevant_filenames) if relevant_filenames else 0


def mrr_at_k(retrieved_docs, relevant_filenames, relevant_keywords, k):
    """
    MRR@k = 1/rank первого релевантного результата.
    """
    for rank, doc in enumerate(retrieved_docs[:k], start=1):
        source = doc.metadata.get("source", "")
        content = doc.page_content.lower()
        if any(rel in source for rel in relevant_filenames):
            return 1 / rank
        if any(kw.lower() in content for kw in relevant_keywords):
            return 1 / rank
    return 0.0


print("Функции метрик определены: hit_at_k, recall_at_k, mrr_at_k")

Функции метрик определены: hit_at_k, recall_at_k, mrr_at_k


In [21]:
K = 5  # top-k для оценки

results = []

for q in EVAL_QUERIES:
    retrieved = vectorstore.similarity_search(q["query"], k=K)

    hit   = hit_at_k(retrieved, q["relevant_docs"], q["relevant_keywords"], k=K)
    rec   = recall_at_k(retrieved, q["relevant_docs"], q["relevant_keywords"], k=K)
    mrr   = mrr_at_k(retrieved, q["relevant_docs"], q["relevant_keywords"], k=K)

    # Детали топ-1 результата
    top1 = retrieved[0] if retrieved else None
    top1_source  = top1.metadata.get("source", "—")  if top1 else "—"
    top1_section = top1.metadata.get("h2",     "—")  if top1 else "—"

    results.append({
        "id":           q["id"],
        "query":        q["query"],
        "hit@k":        hit,
        "recall@k":     round(rec, 2),
        "mrr@k":        round(mrr, 2),
        "top1_source":  top1_source,
        "top1_section": top1_section,
        "expected":     str(q["relevant_docs"]),
        "note":         q["note"],
    })

df_eval = pd.DataFrame(results)

# ── Итоговые метрики ──────────────────────────────────────────
mean_hit    = df_eval["hit@k"].mean()
mean_recall = df_eval["recall@k"].mean()
mean_mrr    = df_eval["mrr@k"].mean()

print(f"\n{'═'*60}")
print(f"  K = {K}")
print(f"  Hit@{K}    = {mean_hit:.3f}   ({int(mean_hit * len(df_eval))}/{len(df_eval)} запросов)")
print(f"  Recall@{K} = {mean_recall:.3f}")
print(f"  MRR@{K}   = {mean_mrr:.3f}")
print(f"{'═'*60}")


════════════════════════════════════════════════════════════
  K = 5
  Hit@5    = 1.000   (10/10 запросов)
  Recall@5 = 1.000
  MRR@5   = 0.925
════════════════════════════════════════════════════════════


In [22]:
display_cols = ["id", "query", "hit@k", "recall@k", "mrr@k", "top1_source"]

def color_hit(v):
    if v == 1:
        return "background-color: #c8e6c9"  # зелёный
    elif v == 0:
        return "background-color: #ffcdd2"  # красный
    return ""

display(
    df_eval[display_cols].style
    .map(color_hit, subset=["hit@k"])
    .format({"recall@k": "{:.2f}", "mrr@k": "{:.2f}"})
)

,id,query,hit@k,recall@k,mrr@k,top1_source
0,Q01,как ухаживать за сухой кожей зимой,1,1.00,0.25,02_spring_skin_prep.md
1,Q02,какие кислоты помогают от акне и расширенных пор,1,1.00,1.00,02_acne_diet.md
2,Q03,что такое постакне и как убрать тёмные пятна,1,1.00,1.00,01_acne_and_post_acne.md
3,Q04,продукты и питание для здоровой кожи,1,1.00,1.00,01_healthy_eating_basics.md
4,Q05,как правильно наносить и выбирать SPF крем,1,1.00,1.00,04_spf_protection.md
5,Q06,ретинол с чего начать и как избежать раздражения,1,1.00,1.00,01_retinol_and_derivatives.md
6,Q07,уход за жирной кожей матирование и поры,1,1.00,1.00,02_oily_skin.md
7,Q08,витамин С сыворотка осветление пигментных пятен,1,1.00,1.00,02_pigmentation.md
8,Q09,выпадение волос причины и что делать,1,1.00,1.00,04_split_ends.md
9,Q10,как читать состав косметики и что такое INCI,1,1.00,1.00,03_reading_inci_labels.md


In [23]:
failures = df_eval[df_eval["hit@k"] == 0]

if failures.empty:
    print("✅ Все запросы получили хотя бы один релевантный результат в top-k")
else:
    print(f"❌ Провальных запросов: {len(failures)}\n")
    for _, row in failures.iterrows():
        print(f"  {row['id']}: {row['query']}")
        print(f"    Ожидалось:  {row['expected']}")
        print(f"    Получено:   {row['top1_source']}  |  {row['top1_section']}")
        print()

        # Показываем top-5 для этого запроса подробно
        retrieved = vectorstore.similarity_search(row["query"], k=K)
        print(f"    Top-{K} результатов:")
        for i, doc in enumerate(retrieved, 1):
            s = doc.metadata.get("source", "—")
            h = doc.metadata.get("h2", doc.metadata.get("h1", "—"))
            score_info = f"{len(doc.page_content)} симв."
            print(f"      {i}. {s:<40} [{h[:40]}]  {score_info}")
        print()



✅ Все запросы получили хотя бы один релевантный результат в top-k


In [24]:
df_eval.to_csv("retrieval_eval_results.csv", index=False, encoding="utf-8")
print("💾 Сохранено: retrieval_eval_results.csv")
print(f"\nИтог: Hit@{K}={mean_hit:.2f}  Recall@{K}={mean_recall:.2f}  MRR@{K}={mean_mrr:.2f}")

💾 Сохранено: retrieval_eval_results.csv

Итог: Hit@5=1.00  Recall@5=1.00  MRR@5=0.93


## Эксперименты

### 2.3.6. Сравнительный эксперимент по параметрам retrieval: top_k = 3 vs top_k = 5

In [25]:
def evaluate_at_k(vectorstore, eval_queries, k):
    """Прогоняет все контрольные запросы и считает метрики при заданном k."""
    hits, recalls, mrrs = [], [], []

    for q in eval_queries:
        retrieved = vectorstore.similarity_search(q["query"], k=k)
        hits.append(hit_at_k(retrieved, q["relevant_docs"], q["relevant_keywords"], k))
        recalls.append(recall_at_k(retrieved, q["relevant_docs"], q["relevant_keywords"], k))
        mrrs.append(mrr_at_k(retrieved, q["relevant_docs"], q["relevant_keywords"], k))

    return {
        "k":        k,
        "hit@k":    round(sum(hits)    / len(hits),    3),
        "recall@k": round(sum(recalls) / len(recalls), 3),
        "mrr@k":    round(sum(mrrs)    / len(mrrs),    3),
    }

k_values = [3, 5]
exp_results = [evaluate_at_k(vectorstore, EVAL_QUERIES, k) for k in k_values]
df_exp = pd.DataFrame(exp_results)

print("Результаты эксперимента: top_k = 3 vs top_k = 5")
print("=" * 45)
print(df_exp.to_string(index=False))
print()

# Дельты
for metric in ["hit@k", "recall@k", "mrr@k"]:
    delta = df_exp[metric].iloc[1] - df_exp[metric].iloc[0]
    sign  = "+" if delta >= 0 else ""
    print(f"  Δ {metric:<10} = {sign}{delta:.3f}  (k=3 → k=5)")

Результаты эксперимента: top_k = 3 vs top_k = 5
 k  hit@k  recall@k  mrr@k
 3    0.9       0.9  0.900
 5    1.0       1.0  0.925

  Δ hit@k      = +0.100  (k=3 → k=5)
  Δ recall@k   = +0.100  (k=3 → k=5)
  Δ mrr@k      = +0.025  (k=3 → k=5)


In [26]:
print("\nПозапросное сравнение (hit = нашёл / не нашёл):")
print(f"  {'ID':<5} {'Запрос (60 символов)':<62} k=3   k=5")
print("  " + "─" * 78)

for q in EVAL_QUERIES:
    r3 = vectorstore.similarity_search(q["query"], k=3)
    r5 = vectorstore.similarity_search(q["query"], k=5)
    h3 = hit_at_k(r3, q["relevant_docs"], q["relevant_keywords"], 3)
    h5 = hit_at_k(r5, q["relevant_docs"], q["relevant_keywords"], 5)
    mark3 = "✅" if h3 else "❌"
    mark5 = "✅" if h5 else "❌"
    changed = " ← изменилось" if h3 != h5 else ""
    print(f"  {q['id']:<5} {q['query'][:60]:<62} {mark3}     {mark5}{changed}")


Позапросное сравнение (hit = нашёл / не нашёл):
  ID    Запрос (60 символов)                                           k=3   k=5
  ──────────────────────────────────────────────────────────────────────────────
  Q01   как ухаживать за сухой кожей зимой                             ❌     ✅ ← изменилось
  Q02   какие кислоты помогают от акне и расширенных пор               ✅     ✅
  Q03   что такое постакне и как убрать тёмные пятна                   ✅     ✅
  Q04   продукты и питание для здоровой кожи                           ✅     ✅
  Q05   как правильно наносить и выбирать SPF крем                     ✅     ✅
  Q06   ретинол с чего начать и как избежать раздражения               ✅     ✅
  Q07   уход за жирной кожей матирование и поры                        ✅     ✅
  Q08   витамин С сыворотка осветление пигментных пятен                ✅     ✅
  Q09   выпадение волос причины и что делать                           ✅     ✅
  Q10   как читать состав косметики и что такое INCI             

Вывод:
  k=3: меньше нерелевантных чанков попадает в контекст LLM,
       но риск пропустить нужный фрагмент выше.
  k=5: recall растёт за счёт дополнительного поиска,
       hit@k и mrr@k стабильны или незначительно меняются.

  Рекомендация: использовать k=5 как рабочее значение,
  так как прирост recall оправдывает небольшое увеличение
  контекста при передаче в LLM.

## Добавление новых документов

In [27]:
NEW_DOCS = {

"03_skincare_by_type_and_concern/02_skin_concerns/03_rosacea_and_couperose.md": """---
title: "Купероз и розацеа: уход и триггеры"
category: "уход_за_кожей"
subcategory: "проблемы_кожи"
tags: ["купероз", "розацеа", "краснота", "сосуды", "чувствительная_кожа", "триггеры"]
skin_type: ["чувствительная"]
skin_concern: ["купероз", "розацеа", "краснота"]
season: "все"
---

# Купероз и розацеа

## Что такое купероз и розацеа

Купероз — стойкое расширение поверхностных сосудов кожи, видимое в виде сеточки или звёздочек. Розацеа — хроническое воспалительное заболевание кожи лица с периодическими обострениями, покраснением и папулами. Оба состояния требуют мягкого, успокаивающего ухода и исключения триггеров.

### Отличия

- Купероз — только сосудистая проблема, без воспаления
- Розацеа — воспалительное заболевание, может включать акне-подобные высыпания
- Оба усиливаются от тепла, алкоголя и агрессивной косметики

---

## Основные триггеры

Знание триггеров — ключ к управлению розацеа и куперозом.

- Температурные перепады: баня, горячий душ, мороз
- Алкоголь, острая пища, кофе
- Агрессивное очищение и физические скрабы
- Стресс и гормональные колебания
- Некоторые ингредиенты: спирт, ментол, эвкалипт

---

## Уход при куперозе и розацеа

### Очищение

- Только мицеллярная вода или кремовый гель при комнатной температуре
- Без паровых процедур и горячей воды
- Промакивать кожу, не тереть

### Активные компоненты

- Азелаиновая кислота 10–15% — снижает воспаление и покраснение
- Ниацинамид 4–5% — укрепляет сосудистую стенку
- Центелла азиатская (CICA) — успокаивает и восстанавливает барьер
- Пантенол, аллантоин — противовоспалительный эффект

### Что исключить

- Ретинол в активной стадии розацеа — усиливает флашинг
- AHA/BHA кислоты — раздражают
- Продукты с отдушками и спиртом

---

## SPF при розацеа

SPF обязателен ежедневно — ультрафиолет является одним из главных триггеров обострений. Предпочтительны минеральные фильтры (оксид цинка, диоксид титана), так как они реже вызывают раздражение.

---

## Когда к дерматологу

- Стадия III–IV розацеа (фима, офтальморозацеа)
- Папулы не проходят при уходе более 4 недель
- Назначение метронидазола, азелаиновой кислоты Rx — только по рецепту
""",

"02_cosmetics_and_ingredients/02_active_ingredients_encyclopedia/05_niacinamide.md": """---
title: "Ниацинамид: себорегуляция, поры и пигментация"
category: "косметика"
subcategory: "активные_ингредиенты"
tags: ["ниацинамид", "витамин_б3", "себорегуляция", "поры", "пигментация", "барьер"]
skin_type: ["жирная", "комбинированная", "все типы"]
skin_concern: ["акне", "поры", "пигментация", "тусклость"]
season: "все"
---

# Ниацинамид

## Что такое ниацинамид

Ниацинамид — водорастворимая форма витамина B3 (никотиновой кислоты). Один из наиболее универсальных активных ингредиентов в косметике: подходит большинству типов кожи, хорошо переносится и сочетается с большинством других активов.

---

## Доказанные эффекты

- Себорегуляция: снижает выработку кожного сала при концентрации 2–4%
- Сужение пор: уменьшает видимость расширенных пор
- Осветление: блокирует передачу меланосом в кератиноциты, снижает пигментацию
- Укрепление барьера: стимулирует синтез церамидов и кератина
- Противовоспалительный эффект: снижает покраснение и раздражение

---

## Концентрации и применение

| Концентрация | Эффект | Кому подходит |
|---|---|---|
| 2–4% | Себорегуляция, противовоспалительный | Жирная, комбинированная |
| 5% | Осветление, укрепление барьера | Все типы |
| 10% | Максимальная себорегуляция | Жирная, акне |

Оптимальная рабочая концентрация — 5%. Выше 10% не даёт дополнительного эффекта и может вызывать покраснение.

---

## Совместимость с другими ингредиентами

- ✅ Хорошо сочетается: ретинол, гиалуроновая кислота, AHA/BHA, витамин С (стабильные формы), SPF
- ⚠️ Миф о конфликте с витамином С — актуален только при очень высоких концентрациях обоих, в реальных формулах несущественен
- ❌ Не смешивать в одном шаге с чистой аскорбиновой кислотой высокой концентрации

---

## Ошибки при использовании

- Ожидать быстрого результата — эффект заметен через 4–8 недель
- Использовать концентрацию выше 10% без необходимости — покраснение без дополнительной пользы
- Путать с ниацином (никотиновой кислотой) — это разные вещества с разным действием
""",

"04_hair_care/02_curly_girl_method.md": """---
title: "Метод Curly Girl: уход за кудрявыми волосами"
category: "уход_за_волосами"
subcategory: "методы_ухода"
tags: ["curly_girl", "кудрявые_волосы", "метод_кудряшки", "co-wash", "гель", "диффузор"]
skin_type: []
skin_concern: []
season: "все"
---

# Curly Girl Method

## Что такое метод Curly Girl

Curly Girl Method (CGM) — система ухода за вьющимися и кудрявыми волосами, разработанная Лоррейн Мэсси. Основана на отказе от сульфатов, силиконов и тепловой укладки для восстановления структуры кудри и снижения фриза.

---

## Основные принципы

- **Без сульфатов** — SLS/SLES разрушают кутикулу и высушивают кудри
- **Без силиконов** (не смываемых без сульфатов) — создают накопление, утяжеляют кудрь
- **Co-wash** — мытьё кондиционером вместо шампуня или мягкий шампунь без сульфатов
- **Техника сжатия** — не расчёсывать сухие волосы, использовать микрофибровое полотенце

---

## Базовая рутина

### Мытьё

1. Мягкий безсульфатный шампунь или co-wash
2. Кондиционер — оставить на 3–5 минут, смыть не полностью
3. Не тереть полотенцем — промокнуть микрофиброй или футболкой

### Укладка

1. На влажные волосы — leave-in кондиционер
2. Крем для кудрей или гель (hold medium–strong)
3. Техника «scrunching» — сжимать волосы снизу вверх
4. Сушка диффузором на низкой температуре или air-dry
5. После высыхания — разбить «cast» (корку от геля) руками

---

## Типы кудрей (система Андре Уокера)

- 2A–2C: волнистые — нуждаются в лёгком увлажнении, не перегружать
- 3A–3C: кудрявые — нужна влага и средний hold
- 4A–4C: афро-кудри — максимальное увлажнение, heavy creams, LOC-метод

---

## Ошибки при переходе на CGM

- Ожидать идеальных кудрей с первой недели — период восстановления 4–12 недель
- Использовать слишком много продуктов — перегрузка утяжеляет кудрь
- Не делать clarifying (глубокое очищение) раз в месяц — накопление продуктов
"""
}

print(f"Подготовлено новых документов: {len(NEW_DOCS)}")
for path in NEW_DOCS:
    print(f"  + {path.split('/')[-1]}")

Подготовлено новых документов: 3
  + 03_rosacea_and_couperose.md
  + 05_niacinamide.md
  + 02_curly_girl_method.md


In [28]:
for rel_path, content in NEW_DOCS.items():
    full_path = KB_DIR / rel_path
    full_path.parent.mkdir(parents=True, exist_ok=True)
    full_path.write_text(content, encoding="utf-8")
    print(f"✅ Записан: {rel_path}")

def load_kb(kb_dir):
    records = []
    for md_file in sorted(Path(kb_dir).rglob("*.md")):
        try:
            post = frontmatter.load(md_file)
        except Exception:
            continue
        meta = post.metadata
        if not meta.get("title") or not meta.get("category"):
            continue
        body = post.content.strip()
        records.append({
            "filename":    md_file.name,
            "title":       meta.get("title", ""),
            "category":    meta.get("category", ""),
            "subcategory": meta.get("subcategory", ""),
            "tags":        meta.get("tags", []),
            "skin_type":   meta.get("skin_type", []),
            "skin_concern":meta.get("skin_concern", []),
            "body":        body,
        })
    return records

def build_chunks(records):
    md_splitter   = MarkdownHeaderTextSplitter(
        headers_to_split_on=[("#","h1"),("##","h2"),("###","h3")],
        strip_headers=False
    )
    char_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    all_chunks = []
    for row in records:
        for chunk in md_splitter.split_text(row["body"]):
            sub = char_splitter.split_documents([chunk]) if len(chunk.page_content) > 1000 else [chunk]
            for sc in sub:
                sc.metadata.update({k: (", ".join(str(i) for i in v) if isinstance(v, list) else str(v or ""))
                                    for k, v in row.items() if k != "body"})
                all_chunks.append(sc)
    return all_chunks

def build_faiss(chunks, embedding_model):
    return FAISS.from_documents(chunks, embedding_model)


# Строим ДО обновления (уже есть — используем vectorstore из предыдущего шага)
vs_before = vectorstore  # индекс из 2.3.4

# Строим ПОСЛЕ обновления
records_new  = load_kb(KB_DIR)
chunks_new   = build_chunks(records_new)
vs_after = FAISS.from_documents(chunks_new, embedding_model)
vs_after.save_local("./faiss_index_v2")

print(f"Чанков до обновления:  {vs_before.index.ntotal}")
print(f"Чанков после обновления: {vs_after.index.ntotal}")
print(f"Добавлено чанков: +{vs_after.index.ntotal - vs_before.index.ntotal}")




✅ Записан: 03_skincare_by_type_and_concern/02_skin_concerns/03_rosacea_and_couperose.md
✅ Записан: 02_cosmetics_and_ingredients/02_active_ingredients_encyclopedia/05_niacinamide.md
✅ Записан: 04_hair_care/02_curly_girl_method.md
Чанков до обновления:  182
Чанков после обновления: 182
Добавлено чанков: +0


In [29]:
test_queries = [
    "купероз и розацеа как лечить и какие триггеры",
    "ниацинамид концентрация для жирной кожи",
    "метод кудряшки curly girl co-wash",
]

print("\nSравнение retrieval ДО и ПОСЛЕ обновления базы знаний")
print("=" * 70)

for query in test_queries:
    r_before = vs_before.similarity_search(query, k=3)
    r_after  = vs_after.similarity_search(query, k=3)

    print(f"\n🔍 {query}")
    print(f"  {'ДО':<35} {'ПОСЛЕ':<35}")
    print(f"  {'─'*34} {'─'*34}")
    for b, a in zip(r_before, r_after):
        b_src = b.metadata.get("source","—")[:32]
        a_src = a.metadata.get("source","—")[:32]
        print(f"  {b_src:<35} {a_src:<35}")



Sравнение retrieval ДО и ПОСЛЕ обновления базы знаний

🔍 купероз и розацеа как лечить и какие триггеры
  ДО                                  ПОСЛЕ                              
  ────────────────────────────────── ──────────────────────────────────
  03_rosacea_and_couperose.md         —                                  
  03_rosacea_and_couperose.md         —                                  
  03_rosacea_and_couperose.md         —                                  

🔍 ниацинамид концентрация для жирной кожи
  ДО                                  ПОСЛЕ                              
  ────────────────────────────────── ──────────────────────────────────
  05_niacinamide.md                   —                                  
  02_oily_skin.md                     —                                  
  03_combination_skin.md              —                                  

🔍 метод кудряшки curly girl co-wash
  ДО                                  ПОСЛЕ                              
  ────

## Mini-RAG

In [30]:
llm = GigaChat(
    credentials="MDE5ZDc0MWItYzQ2Mi03OGNmLWE5YWMtZjBlMGM4OGU1MWU2OjUxMGYxMzViLTA0NzMtNDcwMi05ZjhmLTNkMTZiMzk0NDllMA==",
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.1,
)

In [31]:
# Проверка соединения
test = llm.invoke("Ответь одним словом: 2+2=?")
print(f"✅ LLM работает: {test.content}")

✅ LLM работает: 4


In [32]:
RAG_PROMPT = ChatPromptTemplate.from_template("""Ты — эксперт по уходу за кожей и косметологии.
Отвечай строго на основе предоставленного контекста.
Если контекст не содержит ответа — так и скажи, не придумывай.
Отвечай на русском языке. Ответ должен быть понятным и конкретным.

Контекст:
{context}

Вопрос: {question}

Ответ:""")

print("✅ Промпт определён")

✅ Промпт определён


In [33]:
K_RAG = 5  # top-k фрагментов для контекста

retriever = vs_after.as_retriever(search_kwargs={"k": K_RAG})


def format_context(docs):
    """Собирает чанки в единый контекст с заголовками источников."""
    parts = []
    for i, doc in enumerate(docs, 1):
        title   = doc.metadata.get("title", "—")
        section = doc.metadata.get("h2", doc.metadata.get("h1", ""))
        header  = f"[{i}] {title}" + (f" — {section}" if section else "")
        parts.append(f"{header}\n{doc.page_content.strip()}")
    return "\n\n".join(parts)


def format_sources(docs):
    """Возвращает список уникальных источников для ответа."""
    seen, sources = set(), []
    for doc in docs:
        src = doc.metadata.get("source", "—")
        sec = doc.metadata.get("h2", "")
        key = (src, sec)
        if key not in seen:
            seen.add(key)
            sources.append(f"• {src}" + (f"  [{sec}]" if sec else ""))
    return "\n".join(sources)


# LangChain LCEL-цепочка
rag_chain = (
    {
        "context":  retriever | format_context,
        "question": RunnablePassthrough(),
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print(f"✅ Mini-RAG собран (k={K_RAG})")

✅ Mini-RAG собран (k=5)


In [34]:
def ask(question: str, k: int = K_RAG, verbose: bool = True) -> dict:
    """
    Выполняет полный RAG-цикл:
      1. Retrieval top-k чанков
      2. Сборка контекста
      3. Генерация ответа через LLM
      4. Возврат ответа + источников
    """
    docs     = vs_after.similarity_search(question, k=k)
    context  = format_context(docs)
    sources  = format_sources(docs)

    prompt_value = RAG_PROMPT.invoke({"context": context, "question": question})
    answer       = llm.invoke(prompt_value).content

    result = {"question": question, "answer": answer, "sources": sources, "docs": docs}

    if verbose:
        print(f"\n{'═'*65}")
        print(f"❓ {question}")
        print(f"{'─'*65}")
        print(answer)
        print(f"\n📚 Источники (top-{k}):")
        print(sources)
        print(f"{'═'*65}")

    return result

In [35]:
def show_rag_result(result: dict):
    """Красиво выводит результат функции ask()"""
    
    md = f"""
---
### ❓ Вопрос
{result['question']}

### 💬 Ответ
{result['answer']}

### 📚 Источники
{result['sources']}

---
"""
    display(Markdown(md))

result = ask("Как использовать ниацинамид?", verbose=False)
show_rag_result(result)


---
### ❓ Вопрос
Как использовать ниацинамид?

### 💬 Ответ
Для эффективного использования ниацинамида следуйте таким рекомендациям:

1. **Выберите продукт с правильной концентрацией**: рабочая концентрация ниацинамида составляет 2-10%, и он должен находиться ближе к началу списка INCI, чтобы гарантировать достаточное количество активного ингредиента.
   
2. **Используйте регулярно**: применяйте средство ежедневно утром и вечером, особенно в составе дневного крема или сыворотки.

3. **Начинайте с низкой концентрации**: если вы впервые используете ниацинамид, начните с небольшой дозы (например, 2%) и постепенно увеличивайте до рекомендуемого уровня.

4. **Не ожидайте мгновенного эффекта**: первые улучшения заметны примерно через 4–8 недель регулярного использования.

5. **Избегайте высоких концентраций без необходимости**: использование концентрации свыше 10% без медицинских показаний может вызвать раздражение и покраснение кожи.

6. **Сочетайте с подходящими активными компонентами**: сочетайте ниацинамид с гиалуроновой кислотой, SPF и пептидами меди, избегая одновременного использования с ретиноидами и высокими дозами витамина C.

Соблюдая эти рекомендации, вы сможете эффективно применять ниацинамид для ухода за кожей.

### 📚 Источники
• —  [Правила чтения косметических составов]
• —  [Что такое ниацинамид]
• —  [С чем сочетать и не сочетать]
• —  [Как ввести в уход]
• —  [Ошибки при использовании]

---


In [36]:
demo_queries = [
    "Как ухаживать за кожей с куперозом: какие ингредиенты помогают, а каких избегать?",
    "Ретинол и витамин С — можно ли их использовать вместе?",
    "Какой уход нужен при сухой и чувствительной коже зимой?",
    "Что такое ниацинамид и в какой концентрации его использовать?",
    "Как метод Curly Girl помогает кудрявым волосам?",
]

for q in demo_queries:
    r = ask(q, k=K_RAG,verbose=False)
    show_rag_result(r)


---
### ❓ Вопрос
Как ухаживать за кожей с куперозом: какие ингредиенты помогают, а каких избегать?

### 💬 Ответ
Для ухода за кожей с куперозом важно использовать мягкие, успокаивающие средства, исключающие триггеры, способные спровоцировать обострение.

### Полезные ингредиенты:
- **Гиалуроновая кислота**: удерживает влагу, способствует увлажнению кожи.
- **Церамиды, холестерин, жирные кислоты**: восстанавливают барьерный слой кожи.
- **Масла**: ши, жожоба, авокадо, сквалан — питают кожу, предотвращают сухость и шелушение.
- **Пантенол, глицерин, бетаин**: обеспечивают дополнительное увлажнение и смягчение.
- **Мочевина 5–10%**: размягчает роговой слой, улучшает проникновение активных компонентов.

### Избегать следующие ингредиенты:
- Спирт, ментол, эвкалипт — раздражают сосуды и усиливают покраснения.
- Агрессивные эксфолианты и скрабы — повреждают сосудистую стенку.
- Ароматизаторы и красители — могут вызвать аллергические реакции и раздражение.
- Высокие температуры воды, горячие процедуры — провоцируют расширение сосудов.

Таким образом, правильный выбор ингредиентов и исключение триггеров помогут эффективно управлять состоянием кожи с куперозом.

### 📚 Источники
• —  [Основные триггеры]
• —  [Этапы и принципы очищения кожи]
• —  [Что такое купероз и розацеа]
• —  [Ингредиенты]
• —  [Связь питания и высыпаний]

---



---
### ❓ Вопрос
Ретинол и витамин С — можно ли их использовать вместе?

### 💬 Ответ
Использовать ретинол и витамин С одновременно не рекомендуется, особенно в высоких концентрациях. 

При совместном применении возможны следующие проблемы:
- **Снижение эффективности витамина С**: пептид меди и витамин С конкурируют между собой за рецепторы клеток, что снижает активность каждого компонента.
- **Повышение риска раздражения кожи**: сочетание двух активных ингредиентов увеличивает вероятность возникновения побочных эффектов, таких как раздражение, сухость и покраснение.

Для максимальной пользы лучше применять эти ингредиенты отдельно друг от друга, используя разные этапы ухода или чередуя дни применения. Например, витамин С можно наносить утром, а ретинол — вечером.

### 📚 Источники
• —  [Совместимость с другими ингредиентами]
• —  [Уход при пигментации]
• —  [С чем сочетать и не сочетать]
• —  [Что такое ретинол и как он работает]
• —  [Ключевые активы]

---



---
### ❓ Вопрос
Какой уход нужен при сухой и чувствительной коже зимой?

### 💬 Ответ
При сухой и чувствительной коже зимой необходим комплексный подход к уходу:

### Основные рекомендации:
1. **Интенсивное увлажнение**: 
   - Используйте кремы с высоким содержанием увлажняющих компонентов (гиалуроновая кислота, керамиды, масло ши, алоэ вера).
   
2. **Защита от холода и ветра**:
   - Применяйте специальные защитные средства («cold cream») перед выходом на улицу.
   - Выбирайте кремы с SPF-фильтром для защиты от ультрафиолетового излучения.

3. **Регулярная эксфолиация**:
   - Мягкий пилинг с мелкими частицами раз в неделю поможет удалить ороговевшие клетки и улучшить проникновение активных ингредиентов.

4. **Питание и восстановление барьера**:
   - Включите в уход сыворотки и маски с питательными маслами (авокадо, кокосовое масло), восстанавливающими кожу после воздействия мороза.

5. **Избегайте агрессивных процедур**:
   - Откажитесь от скрабов с крупными абразивными частичками, интенсивного очищения лица горячей водой, длительного пребывания на солнце без защиты.

6. **Консультация дерматолога**:
   - Если состояние кожи ухудшается, появляются признаки атопического дерматита или экземы, обратитесь к специалисту.

7. **Сезонная адаптация**:
   - Переходите постепенно от тяжелых зимних кремов к более легким весенним текстурам, чтобы подготовить кожу к активному солнцу и перепадам температуры.

Таким образом, правильный уход включает регулярное увлажнение, защиту, питание и бережное очищение кожи, особенно в условиях холодного зимнего климата.

### 📚 Источники
• —
• —  [Когда к дерматологу]

---



---
### ❓ Вопрос
Что такое ниацинамид и в какой концентрации его использовать?

### 💬 Ответ
**Ниацинамид** — это водорастворимая форма витамина B3 (никотиновой кислоты), один из наиболее универсальных активных компонентов в косметике. Подходит для большинства типов кожи, хорошо переносится и совместим с большинством других активных веществ.

Рекомендуемая рабочая концентрация ниацинамида составляет **2-10%**. Для эффективного воздействия вещество должно находиться ближе к началу списка ингредиентов (в первой трети или половине состава).

### 📚 Источники
• —  [Правила чтения косметических составов]
• —  [Что такое ниацинамид]
• —  [С чем сочетать и не сочетать]
• —  [Ингредиенты]
• —  [Ошибки при использовании]

---



---
### ❓ Вопрос
Как метод Curly Girl помогает кудрявым волосам?

### 💬 Ответ
Метод Curly Girl (CGM), разработанный Лоррейн Мэсси, направлен на бережный уход за кудрявыми и вьющимися волосами, предотвращая повреждения и сухость. Основные преимущества метода:

- **Отказ от сульфатов**: сульфаты (SLS/SLES) разрушают структуру волоса, нарушая целостность кутикулы и приводя к обезвоживанию и ломкости.
- **Отказ от силиконов**: силиконовые компоненты накапливаются на поверхности волос, утяжеляя их и блокируя доступ влаги внутрь.
- **Использование увлажняющих продуктов**: регулярное применение кондиционеров и кремов для кудрей обеспечивает глубокое увлажнение и защиту волос.
- **Техника scrunching**: сжимание волос снизу вверх позволяет равномерно распределять влагу и фиксирует завитки.
- **Минимизация термической укладки**: использование диффузора на низкой температуре или сушка естественным образом снижает тепловое воздействие.
- **Уход после сушки**: разминание корки («cast») от гелей предотвращает образование жёстких завитков.

Таким образом, метод Curly Girl способствует восстановлению структуры волос, снижению сухости и улучшению внешнего вида кудряшек.

### 📚 Источники
• —  [Что такое метод Curly Girl]
• —  [Базовая рутина]
• —  [Что такое секущиеся концы и почему они появляются]
• —  [Типы кудрей (система Андре Уокера)]
• —  [Основные принципы]

---


## Краткий анализ ошибок

In [37]:
error_cases = [
    {
        "id": "ERR-1",
        "type": "Неполнота базы знаний",
        "query": "Какие процедуры делают при розацеа в салоне: лазер или IPL?",
        "expected": "Подробный ответ про салонные процедуры",
        "problem": "В базе есть документ по розацеа, но он описывает домашний уход. "
                   "Раздел про аппаратную косметологию отсутствует. "
                   "RAG вернёт чанки про домашний уход и честно скажет, "
                   "что инфо нет — это корректное поведение, но полезность низкая.",
        "fix": "Добавить документ про аппаратные процедуры (лазер, IPL, мезотерапия)"
    },
    {
        "id": "ERR-2",
        "type": "Слишком общий запрос → размытый контекст",
        "query": "Расскажи про уход за кожей",
        "expected": "Структурированный ответ по типам кожи",
        "problem": "Запрос не содержит специфики. Embedding-модель не может "
                   "выбрать нужные чанки — retrieval вернёт 5 случайных фрагментов "
                   "из разных документов. LLM получит несвязный контекст и "
                   "либо даст поверхностный ответ, либо начнёт галлюцинировать.",
        "fix": "Добавить в промпт инструкцию запрашивать уточнение, "
               "если вопрос слишком общий. Или использовать query expansion."
    },
    {
        "id": "ERR-3",
        "type": "Смежные темы в одном запросе",
        "query": "Что лучше: ретинол или кислоты для борьбы с акне и морщинами?",
        "expected": "Сравнение двух ингредиентов с учётом двух целей",
        "problem": "Запрос смешивает два ингредиента и два скин-концерна. "
                   "top-5 чанков возьмут по 2-3 из каждого документа. "
                   "Сравнение «ретинол vs кислоты» отдельным документом не представлено, "
                   "поэтому LLM вынуждена синтезировать ответ из разных источников — "
                   "высокий риск неточности.",
        "fix": "Добавить документ с таблицами совместимости ингредиентов. "
               "Альтернатива: multi-hop retrieval — сначала ищем ретинол, "
               "потом кислоты, потом объединяем контекст."
    },
    {
        "id": "ERR-4",
        "type": "Числовой / точный фактический вопрос",
        "query": "Через сколько недель виден результат от ниацинамида?",
        "expected": "4–8 недель",
        "problem": "Точный ответ есть в документе (4–8 недель). "
                   "Если чанкинг разбил этот абзац на границе, нужная цифра "
                   "может оказаться в соседнем чанке, который не попал в top-5. "
                   "Это проблема chunk boundary — решается overlap=100+ "
                   "или хранением перекрывающихся чанков.",
        "fix": "Увеличить overlap до 150-200 символов для числовых фактов. "
               "Или использовать parent-document retriever."
    },
]

print("\n" + "═"*65)
print("  2.3.9. АНАЛИЗ ОШИБОК MINI-RAG")
print("═"*65)

for case in error_cases:
    print(f"\n{case['id']}: {case['type']}")
    print(f"  Запрос:   {case['query']}")
    print(f"  Проблема: {case['problem']}")
    print(f"  Решение:  {case['fix']}")

# Прогоним пограничные запросы через RAG чтобы показать реальный вывод
print("\n── Реальные ответы на пограничные запросы ─────────────────")
for case in error_cases[:2]:
    ask(case["query"], k=K_RAG)


═════════════════════════════════════════════════════════════════
  2.3.9. АНАЛИЗ ОШИБОК MINI-RAG
═════════════════════════════════════════════════════════════════

ERR-1: Неполнота базы знаний
  Запрос:   Какие процедуры делают при розацеа в салоне: лазер или IPL?
  Проблема: В базе есть документ по розацеа, но он описывает домашний уход. Раздел про аппаратную косметологию отсутствует. RAG вернёт чанки про домашний уход и честно скажет, что инфо нет — это корректное поведение, но полезность низкая.
  Решение:  Добавить документ про аппаратные процедуры (лазер, IPL, мезотерапия)

ERR-2: Слишком общий запрос → размытый контекст
  Запрос:   Расскажи про уход за кожей
  Проблема: Запрос не содержит специфики. Embedding-модель не может выбрать нужные чанки — retrieval вернёт 5 случайных фрагментов из разных документов. LLM получит несвязный контекст и либо даст поверхностный ответ, либо начнёт галлюцинировать.
  Решение:  Добавить в промпт инструкцию запрашивать уточнение, если вопрос сли

## Артефакты

In [38]:
eval_rows = []

for q in EVAL_QUERIES:
    retrieved = vs_after.similarity_search(q["query"], k=K)

    hit = hit_at_k(retrieved, q["relevant_docs"], q["relevant_keywords"], k=K)
    rec = recall_at_k(retrieved, q["relevant_docs"], q["relevant_keywords"], k=K)
    mrr = mrr_at_k(retrieved, q["relevant_docs"], q["relevant_keywords"], k=K)

    ret_sources = [doc.metadata.get("source", "—") for doc in retrieved]

    rank = None
    for i, doc in enumerate(retrieved, 1):
        src = doc.metadata.get("source", "")
        content = doc.page_content.lower()
        if any(r in src for r in q["relevant_docs"]) or any(
            kw.lower() in content for kw in q["relevant_keywords"]
        ):
            rank = i
            break

    eval_rows.append({
        "query": q["query"],
        "expected_source": "; ".join(q["relevant_docs"]),
        "retrieved_sources": " | ".join(ret_sources),
        "hit_at_k": hit,
        "recall_at_k": round(rec, 3),
        "mrr_at_k": round(mrr, 3),
        "rank_of_first_relevant": rank if rank else "—",
    })

df_ret = pd.DataFrame(eval_rows)
df_ret.to_csv(f"{ARTIFACTS_DIR}/retrieval_eval.csv", index=False, encoding="utf-8-sig")
print("✅ saved: retrieval_eval.csv")

✅ saved: retrieval_eval.csv


In [39]:
rag_rows = []

demo_queries = [
    "Как ухаживать за кожей с куперозом: какие ингредиенты помогают, а каких избегать?",
    "Ретинол и витамин С — можно ли их использовать вместе?",
    "Какой уход нужен при сухой и чувствительной коже зимой?",
    "Что такое ниацинамид и в какой концентрации его использовать?",
    "Как метод Curly Girl помогает кудрявым волосам?",
]

for question in demo_queries:
    r = ask(question, k=K_RAG, verbose=False)
    docs = r["docs"]
    rag_rows.append({
        "question": r["question"],
        "answer": r["answer"].strip(),
        "retrieved_sources": " | ".join(doc.metadata.get("source", "—") for doc in docs),
    })

df_rag = pd.DataFrame(rag_rows)
df_rag.to_csv(f"{ARTIFACTS_DIR}/rag_examples.csv", index=False, encoding="utf-8-sig")
print(f"✅ saved: rag_examples.csv  ({len(df_rag)} строк)")

✅ saved: rag_examples.csv  (5 строк)


In [40]:
ba_queries = [q["query"] for q in EVAL_QUERIES] + [
    "купероз и розацеа как лечить и какие триггеры",
    "ниацинамид концентрация для жирной кожи",
    "метод кудряшки curly girl co-wash",
]

ba_rows = []

for query in ba_queries:
    r_before = vs_before.similarity_search(query, k=3)
    r_after = vs_after.similarity_search(query, k=3)

    src_before = " | ".join(doc.metadata.get("source", "—") for doc in r_before)
    src_after = " | ".join(doc.metadata.get("source", "—") for doc in r_after)

    ba_rows.append({
        "query": query,
        "before_retrieved_sources": src_before,
        "after_retrieved_sources": src_after,
        "changed": src_before != src_after,
    })

df_ba = pd.DataFrame(ba_rows)
df_ba.to_csv(f"{ARTIFACTS_DIR}/retrieval_before_after_update.csv", index=False, encoding="utf-8-sig")
print(f"✅ saved: retrieval_before_after_update.csv  ({df_ba['changed'].sum()} изменилось)")

✅ saved: retrieval_before_after_update.csv  (13 изменилось)


In [41]:
chunk_rows = []

for i, chunk in enumerate(all_chunks[:50]):
    chunk_rows.append({
        "chunk_id": i,
        "source": chunk.metadata.get("source", "—"),
        "title": chunk.metadata.get("title", "—"),
        "h1": chunk.metadata.get("h1", ""),
        "h2": chunk.metadata.get("h2", ""),
        "h3": chunk.metadata.get("h3", ""),
        "num_chars": len(chunk.page_content),
        "num_words": len(chunk.page_content.split()),
        "preview": chunk.page_content[:120].replace("\n", " "),
    })

df_chunks = pd.DataFrame(chunk_rows)
df_chunks.to_csv(f"{ARTIFACTS_DIR}/chunk_examples.csv", index=False, encoding="utf-8-sig")
print("✅ saved: chunk_examples.csv")

✅ saved: chunk_examples.csv


In [42]:
rank_1_hits = int(
    sum(1 for r in eval_rows if r["rank_of_first_relevant"] == 1)
)

summary = {
    "k": K,
    "num_queries": len(EVAL_QUERIES),
    "hit_at_k": round(df_ret["hit_at_k"].mean(), 3),
    "recall_at_k": round(df_ret["recall_at_k"].mean(), 3),
    "mrr_at_k": round(df_ret["mrr_at_k"].mean(), 3),
    "perfect_hits": int(df_ret["hit_at_k"].sum()),
    "failed_queries": int((df_ret["hit_at_k"] == 0).sum()),
    "rank_1_hits": rank_1_hits,
    "before_after_changed": int(df_ba["changed"].sum()),
    "total_chunks_after": int(vs_after.index.ntotal),
}

with open(f"{ARTIFACTS_DIR}/retrieval_metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("✅ saved: retrieval_metrics_summary.json")
print(json.dumps(summary, ensure_ascii=False, indent=2))

✅ saved: retrieval_metrics_summary.json
{
  "k": 5,
  "num_queries": 10,
  "hit_at_k": 1.0,
  "recall_at_k": 1.0,
  "mrr_at_k": 0.925,
  "perfect_hits": 10,
  "failed_queries": 0,
  "rank_1_hits": 9,
  "before_after_changed": 13,
  "total_chunks_after": 182
}


In [43]:
colors = ["#2ecc71" if h == 1 else "#e74c3c" for h in df_ret["hit_at_k"]]
short_queries = [q[:35] + "…" if len(q) > 35 else q for q in df_ret["query"]]

fig = go.Figure()

fig.add_trace(go.Bar(
    name="Hit@5",
    x=short_queries,
    y=df_ret["hit_at_k"],
    marker_color=colors,
    opacity=0.85,
))
fig.add_trace(go.Scatter(
    name="Recall@5",
    x=short_queries,
    y=df_ret["recall_at_k"],
    mode="lines+markers",
    line=dict(color="#3498db", width=2),
    marker=dict(size=8),
))
fig.add_trace(go.Scatter(
    name="MRR@5",
    x=short_queries,
    y=df_ret["mrr_at_k"],
    mode="lines+markers",
    line=dict(color="#9b59b6", width=2, dash="dot"),
    marker=dict(size=7),
))

fig.update_layout(
    title=(
        f"Retrieval quality per query (k={K})<br>"
        f"<span style='font-size:14px'>"
        f"Hit@{K}={summary['hit_at_k']} | "
        f"Recall@{K}={summary['recall_at_k']} | "
        f"MRR@{K}={summary['mrr_at_k']}"
        f"</span>"
    ),
    xaxis_title="Query",
    yaxis_title="Score",
    yaxis=dict(range=[0, 1.15]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    bargap=0.3,
)

fig.update_xaxes(tickangle=-35)
fig.write_html(f"{ARTIFACTS_DIR}/retrieval_quality_plot.html")
print("✅ saved: retrieval_quality_plot.html")

✅ saved: retrieval_quality_plot.html


In [44]:
for name in [
    "retrieval_eval.csv",
    "rag_examples.csv",
    "retrieval_before_after_update.csv",
    "chunk_examples.csv",
    "retrieval_metrics_summary.json",
    "retrieval_quality_plot.html",
]:
    path = os.path.join(ARTIFACTS_DIR, name)
    size = os.path.getsize(path) if os.path.exists(path) else 0
    print(("✅" if os.path.exists(path) else "❌"), f"{name}  ({size} байт)")

✅ retrieval_eval.csv  (1584 байт)
✅ rag_examples.csv  (10652 байт)
✅ retrieval_before_after_update.csv  (2255 байт)
✅ chunk_examples.csv  (21781 байт)
✅ retrieval_metrics_summary.json  (219 байт)
✅ retrieval_quality_plot.html  (4569497 байт)
